# Churn XAI — Notebook d'exploration

Projet Challenge 1 — Classification Explicable. Cas métier : **Télécom — Attrition client** (Telco Customer Churn).

Ce notebook reproduit le pipeline complet de l'application Streamlit (`app.py`) de façon linéaire,
en réutilisant les mêmes modules (`src/`), pour une inspection rapide sans lancer l'UI.

Pré-requis : exécuter depuis la racine `churn-xai-app/` avec l'environnement virtuel activé
(`pip install -r requirements.txt`).

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.express as px

from src import data as data_mod
from src import prep as prep_mod
from src import models as models_mod
from src import evaluate as eval_mod
from src import explain as explain_mod

2026-11-12 23:25:03.821 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


C:\dev\churn-xai-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Module 1 — Données

In [2]:
df = data_mod.download_telco_dataset()
df = data_mod.clean_telco(df)
print(df.shape)
df.head()

2026-11-12 23:25:07.466 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
target_col = data_mod.DEFAULT_TARGET
data_mod.class_balance(df, target_col)

,count,pct
Churn,,
0,5174,73.46
1,1869,26.54


In [4]:
df.isna().sum()[df.isna().sum() > 0]

TotalCharges    11
dtype: int64

In [5]:
px.histogram(df, x='tenure', color=target_col, barmode='overlay', nbins=30,
             title='Distribution de tenure par churn')

In [6]:
numeric_cols, categorical_cols = prep_mod.split_feature_types(df.drop(columns=[target_col]))
corr = df[numeric_cols + [target_col]].corr(numeric_only=True)
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Corrélations')

## Module 2 — Préparation

In [7]:
feature_cols = [c for c in df.columns if c not in [target_col, 'customerID']]

work_df = df.dropna(subset=[target_col]).copy()
X = work_df[feature_cols]
y = work_df[target_col].astype(int)

numeric_cols, categorical_cols = prep_mod.split_feature_types(X)
preprocessor = prep_mod.build_preprocessor(numeric_cols, categorical_cols, scale=True)

X_train, X_test, y_train, y_test = prep_mod.make_split(X, y, test_size=0.2, stratify=True)
print(f"train={len(X_train)}  test={len(X_test)}  num={len(numeric_cols)}  cat={len(categorical_cols)}")

train=5634  test=1409  num=4  cat=15


## Module 3 — Modélisation et comparaison

In [8]:
trained = {}
for name in models_mod.MODEL_REGISTRY:
    params = models_mod.MODEL_REGISTRY[name]['default_params']
    pipeline, used_params = models_mod.train_model(
        name, preprocessor, X_train, y_train, params, optimize=False
    )
    trained[name] = pipeline
    print(name, '->', used_params)

Regression logistique -> {'C': 1.0}
Arbre de decision -> {'max_depth': 5}


K plus proches voisins -> {'n_neighbors': 15}


Random Forest -> {'n_estimators': 300, 'max_depth': None}


XGBoost -> {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1}


## Module 4 — Évaluation

In [9]:
models_proba = {name: p.predict_proba(X_test)[:, 1] for name, p in trained.items()}
metrics = eval_mod.metrics_table(models_proba, y_test, threshold=0.5)
metrics.sort_values('AUC-ROC', ascending=False)

,AUC-ROC,PR-AUC,precision,rappel,F1
modele,,,,,
Regression logistique,0.8419,0.6323,0.6572,0.5588,0.6040
XGBoost,0.8331,0.6493,0.6300,0.5053,0.5608
Arbre de decision,0.8303,0.6483,0.6347,0.5668,0.5989
K plus proches voisins,0.8215,0.6125,0.5845,0.5642,0.5741
Random Forest,0.8171,0.6058,0.6068,0.4786,0.5351


In [10]:
eval_mod.roc_overlay_figure(models_proba, y_test)

In [11]:
best_model_name = metrics['AUC-ROC'].idxmax()
best_pipeline = trained[best_model_name]
print('Meilleur modèle :', best_model_name)
eval_mod.confusion_matrix_figure(y_test, models_proba[best_model_name], threshold=0.5)

Meilleur modèle : Regression logistique


## Module 5 — Explainable AI (SHAP / LIME)

Explications calculées directement ici (sans passer par l'API FastAPI, pour la version notebook).

In [12]:
feature_names = explain_mod.get_transformed_feature_names(best_pipeline)
native_imp = explain_mod.native_importance(best_pipeline, feature_names)
native_imp.head(15)

tenure                                  1.241319
Contract_Two year                       0.772261
InternetService_DSL                     0.652406
InternetService_Fiber optic             0.640026
MonthlyCharges                          0.596392
Contract_Month-to-month                 0.579304
TotalCharges                            0.516708
PaperlessBilling_No                     0.343260
DeviceProtection_No internet service    0.301082
InternetService_No                      0.301082
OnlineSecurity_No internet service      0.301082
TechSupport_No internet service         0.301082
StreamingTV_No internet service         0.301082
OnlineBackup_No internet service        0.301082
StreamingMovies_No internet service     0.301082
dtype: float64

In [13]:
perm_imp = explain_mod.permutation_importances(best_pipeline, X_test, y_test, n_repeats=5)
perm_imp.head(15)

tenure              0.169406
InternetService     0.039311
Contract            0.032505
MonthlyCharges      0.030623
TotalCharges        0.013128
OnlineSecurity      0.004019
TechSupport         0.003821
StreamingMovies     0.003786
StreamingTV         0.003250
PaymentMethod       0.003095
MultipleLines       0.003002
OnlineBackup        0.001724
PaperlessBilling    0.001566
DeviceProtection    0.001287
SeniorCitizen       0.001110
dtype: float64

In [14]:
shap_explainer = explain_mod.build_shap_explainer(best_pipeline, X_train)
shap_sample = X_test.sample(150, random_state=42)
shap_out = explain_mod.shap_values_for(shap_explainer, best_pipeline, shap_sample)
shap_out.values.shape

(150, 45)

In [15]:
# Explication locale pour un individu du jeu de test
row = X_test.iloc[0]
row_df = X_test.iloc[[0]]
proba = best_pipeline.predict_proba(row_df)[0, 1]

local_shap_out = explain_mod.shap_values_for(shap_explainer, best_pipeline, row_df)
shap_row = pd.Series(local_shap_out.values[0], index=feature_names)

lime_bundle = explain_mod.build_lime_explainer(
    X_train, numeric_cols, categorical_cols, class_names=['No', 'Yes']
)
lime_dict = explain_mod.lime_explanation_for(lime_bundle, best_pipeline, row)

synthesis = explain_mod.synthesis_sentence(shap_row, proba, threshold=0.5)
print(synthesis)
print('\nTop SHAP:', shap_row.abs().sort_values(ascending=False).head(5).index.tolist())
print('Top LIME:', list(lime_dict.keys())[:5])

Ce client a risque de depart faible (probabilite 4.5%), principalement a cause de : tenure (diminue le risque), TotalCharges (augmente le risque), MonthlyCharges (diminue le risque).

Top SHAP: ['tenure', 'TotalCharges', 'MonthlyCharges', 'Contract_Two year', 'Contract_Month-to-month']
Top LIME: ['tenure > 55.00', 'InternetService=Fiber optic', 'TotalCharges > 3835.83', 'Contract=Two year', 'MonthlyCharges > 90.00']


## Conclusion

Voir `README.md` et `one_pager.pdf` à la racine du dépôt pour l'analyse critique complète
(cohérence SHAP/LIME, variables sensibles, compromis performance/interprétabilité, recommandations).